# section 2 — first spectral encoder on GNPS

See `README.md` in this folder for the full plan and reasoning. Quick recap:

- **Phase 0** — get a real GNPS spectral library, load/harmonize it with `matchms`, and interrogate what's actually in it. The open question (what should the model predict?) gets answered here, from the real numbers, not guessed in advance.
- **Phase 1** — binned-spectrum MLP baseline, reusing section 1's `Dataset`/`DataLoader`/`nn.Module`/training-loop patterns. First from-scratch Weights & Biases logging.
- **Phase 2** — continuous peak encoding + a from-scratch tiny transformer encoder. First from-scratch attention implementation. Compared against Phase 1 on the same task and data.


In [ ]:
import math
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from matchms import filtering as msfilters
from matchms import importing as msimport
from torch.utils.data import DataLoader, Dataset

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## Phase 0.1 — get a real GNPS library file

`matchms` does not bundle a training-sized GNPS dataset — its own test fixtures top out at 76 spectra
(and that one is single ion-mode, adduct classes as imbalanced as 53/13/8/1/1). Not usable for training
anything. You need an actual GNPS spectral library export.

GNPS publishes bulk MGF downloads for each of its contributed libraries. Start at the documentation page
(https://ccms-ucsd.github.io/GNPSDocumentation/downloadlibraries/), which points to the library hub
(https://gnps-external.ucsd.edu/gnpslibrary). Pick one moderate-sized, general-purpose library —
avoid a single-instrument or single-study deposit (too narrow, same problem as the test fixtures) and
avoid downloading everything GNPS has pooled together (too large for a laptop CPU exercise).
`GNPS-LIBRARY.mgf`, the original manually-curated library, is a reasonable default if you want a concrete
starting point — but the exact choice doesn't matter much as long as it's a general library with
real size and diversity.

Save the file under `data/gnps/` at the repo root (already gitignored) and point `GNPS_MGF_PATH` at it.


In [ ]:
GNPS_MGF_PATH = Path("../data/gnps/GNPS-LIBRARY.mgf")  # TODO: point this at your downloaded file
assert GNPS_MGF_PATH.exists(), f"no file at {GNPS_MGF_PATH} -- download a GNPS library MGF first"


## Phase 0.2 — load and harmonize with `matchms`

`matchms` represents each spectrum as a `Spectrum` object (peaks + a metadata dict). Raw MGF metadata
keys are inconsistent across depositors (casing, naming, missing fields) — `matchms` ships a
`default_filters` pipeline that harmonizes the common fields (ion mode, adduct, charge, etc.) so you're
not hand-rolling that cleanup yourself.

Look up: `matchms.importing.load_from_mgf` (returns a generator of raw `Spectrum` objects) and
`matchms.filtering.default_filters` (takes one `Spectrum`, returns a harmonized one). What do you do with
a `Spectrum` that `default_filters` can't salvage (e.g. ends up with no peaks)?


In [ ]:
def load_gnps_spectra(path: Path) -> list:
    """Load spectra from an MGF file and run matchms's default harmonization filters."""
    pass


spectra = load_gnps_spectra(GNPS_MGF_PATH)
print(f"loaded {len(spectra)} harmonized spectra")


## Phase 0.3 — interrogate the metadata

Before deciding what the model should predict, look at what's actually here. Use `metadata_value_counts`
below on whatever fields you think matter (`ionmode`, `adduct`, `charge`, `instrument_type`... — check
`spectra[0].metadata.keys()` to see what survived harmonization) and `plot_peak_count_histogram` to see
spectrum complexity. Also worth checking: precursor m/z range — relevant for choosing bin widths and an
m/z cutoff later.


In [ ]:
def metadata_value_counts(spectra, key):
    """Infra: tally how often each value of a metadata field appears across spectra."""
    counts = Counter(s.get(key) for s in spectra)
    return counts.most_common()


def plot_peak_count_histogram(spectra):
    """Infra: distribution of number of peaks per spectrum."""
    n_peaks = [len(s.peaks.mz) for s in spectra]
    plt.figure(figsize=(6, 4))
    plt.hist(n_peaks, bins=40)
    plt.xlabel("peaks per spectrum")
    plt.ylabel("count")
    plt.title("peak count distribution")
    plt.show()
    return pd.Series(n_peaks).describe()


In [ ]:
# Explore. Call metadata_value_counts on the fields you think matter, and look at the peak
# count / precursor m/z distributions. This is the evidence the open question below gets answered from.


## Open question — answer it here

Given what you found above: what will the Phase 1/2 target be? Revisit the candidates in `README.md`
(adduct-type classification, precursor m/z regression, ion-mode classification, or something else the
data surfaced) against the real class sizes/balance you just measured, and commit to one.

**Your answer:**

*(write your reasoning and decision here)*


## Phase 1 — binned-spectrum MLP baseline

Turn each spectrum into a fixed-length vector: divide the m/z axis into fixed-width bins up to some
`mz_max`, and fill each bin with the intensity of the peak(s) landing in it. Two design decisions are
yours: what happens when more than one peak lands in the same bin (sum? take the max?), and what
`mz_max` makes sense given the precursor m/z range you saw in Phase 0.

Look up: `numpy.histogram` (or a manual bin-index computation) — either is fine, but you should be able
to explain what determines the output vector's length given `bin_width` and `mz_max`.


In [ ]:
def bin_spectrum(spectrum, bin_width: float, mz_max: float) -> np.ndarray:
    """Turn a spectrum's peaks into a fixed-length binned-intensity vector."""
    pass


## Phase 1 — Dataset

Same shape as section 1's `Dataset`: `__init__` stores what you need, `__len__` returns the number of
spectra, `__getitem__` returns one `(binned_vector, target)` pair. The target depends on what you decided
above — a class index for classification, a float for regression.


In [ ]:
class BinnedSpectrumDataset(Dataset):
    def __init__(self, spectra, bin_width: float, mz_max: float):
        pass

    def __len__(self):
        pass

    def __getitem__(self, idx):
        pass


## Phase 1 — model

An MLP, same pattern as section 1: a couple of `nn.Linear` layers with a nonlinearity between them. The
input dimension depends on your binning; the output depends on your task (one logit per class, or a
single regression output).


In [ ]:
class BinnedMLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 128):
        super().__init__()
        pass

    def forward(self, x):
        pass


## Phase 1 — training loop, with Weights & Biases

Same train/validate ceremony as section 1. New this section: log metrics to `wandb` instead of (or
alongside) printing them. Look up: `wandb.init`, `wandb.log`, `wandb.finish` (the quickstart guide covers
all three). Keep `train_one_epoch` / `evaluate` generic enough to reuse for Phase 2's model too.


In [ ]:
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    pass


def evaluate(model, dataloader, loss_fn, device, metric_fn):
    pass


## Phase 1 — bin-width sweep

The actual experiment: train the same model across several bin widths and watch your chosen metric move.
This cell is infra — it calls the functions you just wrote across a list of bin widths and collects
the results. Fill in `BIN_WIDTHS` with a reasonable range (from very coarse to fairly fine) given the
precursor m/z range you found in Phase 0.


In [ ]:
BIN_WIDTHS = [...]  # TODO: choose a sensible sweep, e.g. [2.0, 1.0, 0.5, 0.1, 0.05]
MZ_MAX = None  # TODO: set from your Phase 0 precursor m/z interrogation


def run_bin_width_sweep(spectra, bin_widths, mz_max, make_datasets_fn, train_fn, eval_fn):
    """Infra: train + evaluate one model per bin width, return a results DataFrame."""
    results = []
    for bin_width in bin_widths:
        # TODO: build datasets/dataloaders at this bin_width, train a fresh model, evaluate it
        pass
    return pd.DataFrame(results, columns=["bin_width", "metric"])


def plot_bin_width_sweep(results_df):
    plt.figure(figsize=(6, 4))
    plt.plot(results_df["bin_width"], results_df["metric"], marker="o")
    plt.xlabel("bin width (Da)")
    plt.ylabel("metric")
    plt.title("resolution loss: metric vs. bin width")
    plt.gca().invert_xaxis()
    plt.show()


## Phase 2 — continuous peak encoding + a tiny transformer encoder

Same task, same data, no binning. Each spectrum is a *set* of `(m/z, intensity)` pairs — variable
length, no natural order. This is the representation DreaMS (section 3) and MIST (section 4) both build
on, and your first from-scratch attention implementation.


### Continuous positional encoding for m/z

The original Transformer's positional encoding is a set of `sin`/`cos` waves at different frequencies,
applied to an integer position index (Vaswani et al., 2017, section 3.5). Nothing about that formula
actually requires an integer — it's a function of a number.

Guiding question: if you feed a continuous m/z value into that same formula instead of an integer index,
what do you get? Why do you need many frequencies (not just one `sin(mz)`) for the network to be able to
tell nearby m/z values apart?


In [ ]:
def continuous_positional_encoding(values: torch.Tensor, d_model: int) -> torch.Tensor:
    """values: shape (..., ) of continuous scalars (e.g. m/z). Returns shape (..., d_model)."""
    pass


### Peak embedding

Combine each peak's positional-encoded m/z with a projection of its intensity into one `d_model`-dim
vector per peak. How you combine them (add? concatenate then project?) is your call — be ready to
justify it.


In [ ]:
class PeakEmbedding(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        pass

    def forward(self, mz: torch.Tensor, intensity: torch.Tensor) -> torch.Tensor:
        pass


### Scaled dot-product self-attention, from scratch

The formula: `softmax(Q K^T / sqrt(d_k)) V`. Implement it directly with tensor ops (no
`nn.MultiheadAttention`, no `nn.TransformerEncoderLayer`) — that's the point of this section. Wrap it
in a minimal encoder layer: attention, residual connection, layer norm, a small feedforward block, another
residual + norm.

Real gotcha: spectra have different numbers of peaks, so a batch needs padding — which means padded
positions must not be attended to. Look up: "attention mask" / "padding mask". What happens to your
softmax if you don't mask padding out?


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    pass


class SpectrumEncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        pass

    def forward(self, x, mask=None):
        pass


### Pooling and the full encoder

Self-attention gives you one vector per peak. You need one vector per *spectrum* to feed a classification
or regression head — some permutation-invariant reduction over the (unmasked) peak dimension. Mean
pooling over real (non-padded) peaks is the simplest option; a learned attention-pooling or CLS-token is
another. Pick one and be ready to say why.


In [ ]:
class SpectrumEncoder(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, n_layers: int, output_dim: int):
        super().__init__()
        pass

    def forward(self, mz, intensity, mask=None):
        pass


## Phase 2 — Dataset and collation

`__getitem__` here returns a variable-length set of `(mz, intensity)` pairs plus a target — you can't
stack variable-length tensors into a batch directly. Look up: `collate_fn` (the `DataLoader` argument) and
`torch.nn.utils.rnn.pad_sequence`. Your collate function needs to produce the padding mask
`SpectrumEncoderLayer` expects.


In [ ]:
class PeakSetDataset(Dataset):
    def __init__(self, spectra):
        pass

    def __len__(self):
        pass

    def __getitem__(self, idx):
        pass


def collate_peak_sets(batch):
    pass


## Phase 2 — train it

Reuse `train_one_epoch` / `evaluate` from Phase 1 if they're generic enough; adapt if not. Log to `wandb`
under a run name that makes it obvious this is the continuous-encoding model, so you can compare runs in
the wandb dashboard.


In [ ]:
# Adapt your Phase 1 train_one_epoch / evaluate for this model + dataloader, then train it.


## Compare Phase 1 vs Phase 2

Infra cell — plots your bin-width sweep curve against the continuous-encoding model's single result as
a reference line. If your task is regression, this also draws the theoretical quantization-error floor
(`bin_width / sqrt(12)`, the RMSE of a uniform quantizer) for each bin width, so you can see whether your
binned model is actually hitting the floor the *encoding* imposes, or is losing more than that to something
else.


In [ ]:
def plot_phase_comparison(results_df, continuous_metric, is_regression: bool):
    plt.figure(figsize=(6, 4))
    plt.plot(results_df["bin_width"], results_df["metric"], marker="o", label="Phase 1: binned")
    plt.axhline(continuous_metric, color="tab:red", linestyle="--", label="Phase 2: continuous encoding")
    if is_regression:
        quantization_floor = results_df["bin_width"] / math.sqrt(12)
        plt.plot(results_df["bin_width"], quantization_floor, linestyle=":", color="gray",
                 label="quantization floor (bin_width / sqrt(12))")
    plt.xlabel("bin width (Da)")
    plt.ylabel("metric")
    plt.gca().invert_xaxis()
    plt.legend()
    plt.title("Phase 1 vs Phase 2")
    plt.show()


## Reflection

- **What I built:**
- **What I learned:**
- **Where I was stuck, and for how long:**
